# Notebook 05 — PawPrep Demo Walkthrough

Sets up the environment, verifies the system, and launches the PawPrep Gradio interface.

1. Pull latest code and run Colab setup
2. Verify GPU — download Oxford dataset if not already present
3. Smoke test: generate one illustration before launching the UI
4. Launch the public Gradio URL

> **Run on Colab (T4/V100/A100).** Each generation run takes ~30–40s on T4.

## 0 — Colab Bootstrap

In [ ]:
!git clone https://github.com/reddy-nithin/stable-diffusion
!git -C /content/stable-diffusion pull
%run /content/stable-diffusion/scripts/colab_setup.py

## 1 — Download Oxford Dataset (if not already present)

Needed for ControlNet-seg conditioning using Oxford reference images.  
Skip if you already ran `scripts/download_data.py` in a previous notebook.

In [ ]:
from pathlib import Path

oxford_dir = Path('data/oxford-iiit-pet')
if oxford_dir.exists():
    print(f'Dataset already present at {oxford_dir} ✓')
else:
    print('Downloading Oxford-IIIT Pet dataset (~800 MB)…')
    !python scripts/download_data.py
    print('Download complete ✓')

## 2 — Verify GPU

In [ ]:
import torch

device = ('cuda' if torch.cuda.is_available()
          else 'mps' if torch.backends.mps.is_available()
          else 'cpu')
print(f'Device : {device}')
if device == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
elif device == 'cpu':
    print('⚠️  WARNING: CPU only — generation will be very slow (~10 min per image).')
    print('   Switch to a GPU runtime in Colab: Runtime → Change runtime type → T4 GPU.')

## 3 — Pre-flight Smoke Test

Generate **one** image (fastest possible path) before launching the full UI.  
If this fails, debug here rather than during a live recording.

In [ ]:
from src.app.gradio_app import generate
import matplotlib.pyplot as plt

print('Running smoke test (1 variant, seed=42, beagle / cone_collar / clinic)…')

imgs, pos, neg, ctrl, status = generate(
    animal_type='dog',
    breed='beagle',
    condition='cone_collar',
    environment='clinic',
    style='veterinary illustration',
    use_controlnet=True,
    seed=42,
    uploaded_image=None,
    n_variants=1,   # fast — only 1 image
)

print(f'Status  : {status}')
print(f'Prompt  : {pos[:100]}…')

fig, axes = plt.subplots(1, 2 if ctrl else 1, figsize=(10, 4))
if ctrl:
    axes[0].imshow(ctrl); axes[0].set_title('Control image'); axes[0].axis('off')
    axes[1].imshow(imgs[0]); axes[1].set_title('Generated'); axes[1].axis('off')
else:
    axes.imshow(imgs[0]); axes.set_title('Generated (no ControlNet)'); axes.axis('off')
plt.tight_layout()
plt.show()
print('Smoke test passed ✓')

## 4 — Launch the Gradio App

The cell below starts the server and prints a **public `gradio.live` URL**.  
Open that URL in your browser to use the demo.  
**Copy the URL now** — you'll need it for the video recording.

In [ ]:
from src.app.gradio_app import build_interface

demo = build_interface(data_root='data')

demo.launch(
    share=True,          # generates a public gradio.live URL
    show_error=True,
    quiet=False,
)

# ⬆️  The public URL is printed above. Open it to begin recording.

---

## 5 — 90-Second Demo Script

Use this script for the video recording. Practice once before hitting record.

---

### 🎬 Recording script (~90 seconds)

**[0:00 – 0:10] — Introduction**  
*Show the app title + disclaimer banner*  
> "This is our AI Animal Care system — it turns structured veterinary inputs into educational illustrations using Stable Diffusion 1.5 with ControlNet conditioning."

---

**[0:10 – 0:25] — First generation (ControlNet ON, seg default)**  
*Set: Dog | Beagle | cone_collar | clinic | veterinary illustration | ControlNet ON | seed=42*  
*Click Generate*  
> "The breed, condition, environment, and style all map into a structured prompt. With ControlNet on, we use a real Oxford-IIIT Pet photo of a Beagle as the shape guide — you can see the seg map here."

---

**[0:25 – 0:45] — Compare structured vs naive (note the prompt text)**  
*Open the 'Resolved prompts' accordion and read the positive prompt aloud*  
> "Notice the structured prompt: style, breed, species, condition clause, environment clause, lighting, lens — all from our YAML taxonomy. This is cell D in our 2×2 ablation."

---

**[0:45 – 1:05] — Second generation (upload your own image)**  
*Upload a pet photo | Keep ControlNet ON | switch breed to match your photo | click Generate*  
> "If you have your own pet's photo, you can upload it. The system extracts Canny edges and uses them as conditioning — so your pet's pose and shape guide the generation."

---

**[1:05 – 1:20] — Turn ControlNet OFF / change condition (ablation showcase)**  
*Uncheck ControlNet | change condition to health_exam | click Generate*  
> "Without ControlNet we're in cell C — structured prompt, no shape conditioning. Notice how the pose varies more freely across seeds, while the condition is still captured by the structured prompt."

---

**[1:20 – 1:30] — Close**  
> "All images include the ethics disclaimer — these are educational illustrations only, not clinical guidance. The full pipeline includes CLIPScore, DINOv2, and LPIPS evaluation confirming cell D — structured prompt plus ControlNet — outperforms the naive baseline across all three metrics."

---

### 🎯 Tips for a clean recording
- Use a screen recorder that captures browser + audio (OBS, Loom, or Colab's built-in record feature)
- Run the smoke test (§3) before recording so models are already cached — generations will be faster
- Zoom the browser to 90% so the full UI is visible without scrolling
- Start recording *after* the Gradio URL is open and the page has fully loaded

## 6 — After Recording: Shutdown

In [ ]:
# Run this cell to gracefully shut down the Gradio server after recording
demo.close()
print('Gradio server stopped.')

---
**End of Notebook 05** — Phase 6 complete.  

Next → Phase 7: `README.md`, `docs/slides_outline.md`, `docs/ETHICS.md`, and final submission tag `v1.0-submission`.